# Fashion-MNIST Neural Network

Clothing classifier built using TensorFlow/Keras — following along with the O'Reilly *Deep Learning* book.

**Architecture:** `784 inputs (flattened 28×28) → tanh hidden layer (100 neurons) → softmax → 10 classes`

**Techniques:** TensorFlow/Keras high-level API · Dense layers · Tanh + Softmax activations · Sparse categorical crossentropy loss · Adam optimiser

**Classes:** T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

Fashion-MNIST has the same shape as the original MNIST dataset (28×28 grayscale images, 60,000 train / 10,000 test, 10 classes) but is designed to be a harder, more realistic benchmark. The goal here is to rebuild the MNIST classifier but using Keras instead of writing the forward/backward pass by hand — and then later extend it into a proper CNN with pooling.


In [4]:
%pip install tensorflow 


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
# import dataset
from keras.datasets import mnist

# import tensorflow, numpy and matplotlib
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# for cleaner outputs
import sys


---
## Loading the Data

`fashion_mnist.load_data()` returns two tuples — one for training, one for testing:

- **`x_train`** — a 3D NumPy array of shape `(60000, 28, 28)`: 60,000 images, each a 28×28 grid of pixel values (0–255)
- **`y_train`** — a 1D array of shape `(60000,)`: the correct class label (0–9) for each training image, where each number represents a clothing item (T-shirt/top, Trouser, Pullover, etc.)
- **`x_test`** / **`y_test`** — same structure but 10,000 images, kept separate to evaluate the trained network

So `x_train[0]` is the first image (a 28×28 array of numbers) and `y_train[0]` is the class label for what it shows.

```
x_train = [
    [[...28 pixels...], [...28 pixels...], ... 28 rows],  ← image 0
    [[...28 pixels...], [...28 pixels...], ... 28 rows],  ← image 1
    ...
    [[...28 pixels...], [...28 pixels...], ... 28 rows],  ← image 59,999
]
```

60,000 "rows", each one being a complete image represented as a 28×28 array.


In [41]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Human-readable class names for Fashion-MNIST labels 0-9
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

training_image_count, training_image_height, training_image_width = x_train.shape[0], x_train.shape[1], x_train.shape[2]
training_label_count, training_label_dtype = y_train.shape[0], y_train.dtype
print(f"Training Data:")
print(f"Training Images: {training_image_count}")
print(f"Each image is {training_image_height} x {training_image_width} pixels; totaling {training_image_height * training_image_width} pixels")
print(f"Training Labels: {training_label_count}")
print(f"Label values: 0-9 (dtype: {training_label_dtype}) — one of: {', '.join(class_names)}")

test_image_count, test_image_height, test_image_width = x_test.shape[0], x_test.shape[1], x_test.shape[2]
test_label_count, test_label_dtype = y_test.shape[0], y_test.dtype
print(f"\nTest Data:")
print(f"Test Images: {test_image_count}")
print(f"Each image is {test_image_height} x {test_image_width} pixels; totaling {test_image_height * test_image_width} pixels")
print(f"Test Labels: {test_label_count}")
print(f"Label values: 0-9 (dtype: {test_label_dtype})")


Training Data:
Training Images: 60000
Each image is 28 x 28 pixels; totaling 784 pixels
Training Labels: 60000
Label values: 0-9 (dtype: uint8) — one of: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

Test Data:
Test Images: 10000
Each image is 28 x 28 pixels; totaling 784 pixels
Test Labels: 10000
Label values: 0-9 (dtype: uint8)


---
## Network Parameters

Before building the network we define a few key values that control its size and how it learns:

- **`alpha`** — the learning rate. Controls how big a step the network takes when updating its weights. Too high and it overshoots; too low and it learns very slowly. Note: when using Keras with the Adam optimiser, this is managed internally — we don't need to pass it explicitly.
- **`input_size`** — 784 (28×28). Each image is flattened into a single row of 784 pixel values to feed into the first layer.
- **`hidden_size`** — 100. The number of neurons in the hidden layer. More neurons can learn more complex patterns but cost more compute and can overfit.
- **`output_size`** — 10. One neuron per class (clothing category 0–9).
- **`epochs`** — how many full passes we make over the training data.


In [31]:
# ── Hyperparameters ───────────────────────────────────────────────────────
input_size  = (training_image_height , training_image_width)  # (28, 28)
hidden_size = 100
output_size = 10
epochs      = 20


---
## Building the Network

Unlike the Grokking notebook where we wrote forward propagation, backpropagation and the weight updates by hand, here we use TensorFlow/Keras' high-level API. Keras builds the weight matrices for us and handles training through the `fit()` method.

Two things to note before defining the model:

1. **Normalising pixel values.** The raw pixel values are integers in the range 0–255. Dividing by 255.0 rescales them to floats in the range 0–1, which helps the network learn more stably (activations don't blow up and gradients stay well-scaled).
2. **Flattening happens inside the network.** We don't need to flatten the images ourselves like we did in the NumPy version — Keras' `Flatten` layer does it as part of the forward pass.

We also use a subset (1,000 training / 1,000 test images) here to keep iteration fast while experimenting. Increase these numbers to get better accuracy once the architecture is settled.


In [42]:
# Use a subset of the data so training is fast while experimenting
num_train = 1000
num_test  = 100
batch_size = 10

# Normalise pixel values from 0-255 to 0-1
training_images = x_train[0:num_train] / 255.0
training_labels = y_train[0:num_train]
test_images     = x_test[0:num_test]   / 255.0
test_labels     = y_test[0:num_test]

# ── Build the model ───────────────────────────────────────────────────────
model = tf.keras.models.Sequential([
    # Declare the shape of each input image: 28 x 28 grayscale
    tf.keras.layers.Input(shape=(training_image_height, training_image_width)),

    # layer_0: flatten the 28x28 image into a 784-length vector
    tf.keras.layers.Flatten(),

    # layer_1: hidden layer — 100 neurons with tanh activation
    tf.keras.layers.Dense(hidden_size, activation=tf.nn.tanh),

    # dropout: randomly zero 20% of the hidden activations during training                                                                                                                                                                        
    tf.keras.layers.Dropout(0.5),

    # layer_2: output layer — 10 neurons with softmax, one per class
    tf.keras.layers.Dense(output_size, activation=tf.nn.softmax),
])

# ── Compile and train ─────────────────────────────────────────────────────
# - optimizer: Adam — an adaptive gradient descent variant that picks a good learning rate per weight
# - loss: sparse_categorical_crossentropy — for integer class labels (0-9) rather than one-hot vectors
# - metrics: accuracy — shown per epoch so we can see training progress
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Train the network on the training images/labels for `epochs` iterations.
# verbose=2 prints one clean line per epoch instead of the live progress bar.
print("Training…\n")
history = model.fit(training_images, training_labels, epochs=epochs, batch_size = 100, verbose=0)

# ── Evaluate on the test set ─────────────────────────────────────
print("\nEvaluating on test set…")
test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=1)

# ── Clean summary ─────────────────────────────────────────────────────────
final_train_accuracy = history.history["accuracy"][-1]
final_train_loss     = history.history["loss"][-1]

print("\n── Results ───────────────────────────────────────────────")
print(f"Final training accuracy : {final_train_accuracy * 100:.1f}%  (loss {final_train_loss:.3f})")
print(f"Test accuracy           : {test_accuracy       * 100:.1f}%  (loss {test_loss:.3f})")
print(f"Train/test gap          : {(final_train_accuracy - test_accuracy) * 100:+.1f}%")


# exploring model outputs
classifications = model.predict(test_images)
print(classifications[0])
print(test_labels[0])


Training…


Evaluating on test set…
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9000 - loss: 0.3610 

── Results ───────────────────────────────────────────────
Final training accuracy : 95.0%  (loss 0.232)
Test accuracy           : 90.0%  (loss 0.361)
Train/test gap          : +5.0%
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
[2.0294590e-04 1.8102348e-05 3.4540522e-04 1.6999317e-03 2.9929526e-05
 3.1353673e-05 1.2253732e-05 9.9460059e-01 9.6498778e-05 2.9629483e-03]
7
